# Generation of Raw Data Dataset with Annotations

This notebook is designed to curate a dataset of chemical standards (EMA) for mass spectrometry. It performs the following key tasks:

1.  **Ingests annotation files** for C18 and HILIC chromatography.
2.  **Validates and corrects file paths** for raw data (`.mzML`) files.
3.  **Expands the dataset** to include various Collision Energies (CE) available for each standard.
4.  **Extracts MS2 spectral data** from pre-processed HDF5 files based on precursor m/z and retention time.
5.  **Compiles the final dataset** into a Parquet file and organizes the raw data for distribution.

In [ ]:
import pandas as pd
import os
import multiprocessing as mp
import subprocess


## 1. Load and Preprocess Annotation Files

We start by defining the curation level and input paths. The code loads TSV files containing standards data for both Positive and Negative polarities across C18 and HILIC columns. 

**Key steps:**
* Standardize column names (handling differences between 'name' and 'compound_name').
* Infer `chromatography` type based on the filename.
* **Polarity Inference:** A fallback logic fills in missing polarity values based on the filename string (e.g., `_POS_` vs `_NEG_`).

In [ ]:

curation_level = 'production'  # 'everything' or 'production'
destination_folder = '/pscratch/sd/b/bpb/ema_standards_rawdata/'

cols = ['label', 'adduct', 'polarity', 'mz',
       'rt_peak', 'rt_min', 'rt_max', 'file_name','inchi']
ema_files = [f'../C18/{curation_level}/C18_EMA-standards_positive.tsv',
             f'../C18/{curation_level}/C18_EMA-standards_negative.tsv',
                f'../HILIC/{curation_level}/HILIC_EMA-standards_positive.tsv',
                f'../HILIC/{curation_level}/HILIC_EMA-standards_negative.tsv']
df_list = []
for f in ema_files:
    if 'HILIC' in os.path.basename(f):
        usecols = cols + ['compound_name']
    else:
        usecols = cols + ['name']
    temp = pd.read_csv(f, sep='\t', usecols=cols)
    temp['chromatography'] = 'hilic' if 'HILIC' in os.path.basename(f) else 'c18'
    df_list.append(temp)
df = pd.concat(df_list, ignore_index=True)
df = df[pd.notna(df['file_name'])]

# there is a block where nobody put in the polarity
for i,row in df[pd.isna(df['polarity'])].iterrows():
    if '_POS_' in row['file_name']:
        df.at[i,'polarity'] = 'positive'
    elif '_NEG_' in row['file_name']:
        df.at[i,'polarity'] = 'negative'
    else:
        raise ValueError(f'Cannot infer polarity for file {row["file_name"]}')

df.reset_index(drop=True, inplace=True)

df.head()

## 2. Index Available Old Raw Data Files That are unspecified

To ensure we can locate the raw data files (which may have moved), we first index all `.mzML` files in the target directory (`mzml_dir`).

* We use the Linux `find` command via `subprocess` because it is significantly faster than Python's `os.walk` for large directory trees.
* The result is stored in `mzml_file_dict` for O(1) lookup speed during the validation step.

In [ ]:
# get full filenames of all mzML files in '/global/cfs/cdirs/metatlas/raw_data/akuftin'
mzml_dir = '/global/cfs/cdirs/metatlas/raw_data/akuftin'
# use linux find since it is so much faster than os.walk
result = subprocess.run(['find', mzml_dir, '-name', '*.mzML'], stdout=subprocess.PIPE)
all_mzml_files = result.stdout.decode('utf-8').splitlines()
# convert to a dict with just the basename as key for easy lookup
mzml_file_dict = {os.path.basename(f): f for f in all_mzml_files}

## 3. Validate and Locate Files

The paths in the annotation TSVs might be outdated. This section iterates through the unique filenames required by our dataset and checks if they exist.

* **Hit:** If the file exists, we keep it.
* **Miss:** If the file is missing, we check our pre-built `mzml_file_dict`.
* **Fallback:** If it's not in the dict, we run a targeted system `find` command as a last resort.
* Finally, we update the dataframe with the correct, absolute paths.

In [ ]:
files = df['file_name'].unique().tolist()
# make sure each file exists
replace_map = {}
for f in files:
    if not os.path.exists(f):
        # run a find command to locate it
        # see if it is in the mzml_file_dict first
        if os.path.basename(f) in mzml_file_dict:
            found_file = mzml_file_dict[os.path.basename(f)]
            print(f'Replacing {f} with {found_file}')
            replace_map[f] = found_file
        else:
            dir = '/global/cfs/cdirs/metatlas/raw_data/akuftin'
            result = os.popen(f'find {dir} -name "{os.path.basename(f)}"').read().strip()
            if result:
                replace_map[f] = result
            else:
                print(f'File {f} not found!')
                break
df['file_name'] = df['file_name'].replace(replace_map)

# Double check that all files exist

In [ ]:
files = df['file_name'].unique().tolist()
for f in files:
    assert os.path.exists(f), f'File {f} does not exist!'
    assert os.path.exists(f.replace('.h5','.mzML')), f'File {f} does not exist as mzML!'

## 4. Expand Dataset with Collision Energies (CE)

Mass spectrometry standards are often run at multiple Collision Energies (e.g., 10, 20, 40 eV). The annotation file might point to one file, but we want to capture *all* CE variations associated with that sample.

* `find_ce_group`: This worker function identifies the "prefix" of a file (removing the specific CE tag) and searches for sibling files with different CE tags (e.g., finding `_CE20.h5` and `_CE40.h5` given `_CE10.h5`).
* We use `multiprocessing` to run this search in parallel across the filesystem.

In [ ]:
import os
import multiprocessing as mp
import pandas as pd # Assuming pandas is needed for the initial df input

# Worker function
def find_ce_group(f):
    file_parts = os.path.basename(f).split('_')
    file_dir = os.path.dirname(f)
    
    # Handle case where file_dir is empty (file in current directory)
    search_dir = file_dir if file_dir else '.'

    ce_part = [(i, part) for i, part in enumerate(file_parts) if '-CE' in part]
    
    if ce_part:
        # Reconstruct the prefix: everything before the CE part
        file_part = '_'.join(file_parts[:ce_part[0][0]])
        
        # Construct find command
        # Note: Using '-maxdepth 1' is usually safer to avoid scanning subdirectories recursively, 
        # but I have kept your original logic intact.
        find_command = f'find {search_dir} -name "{file_part}_*-CE*.h5"'
        
        # Execute command
        result = os.popen(find_command).read().strip()
        ce_files = result.splitlines()
        
        # Optional: Print status (Note: prints can overlap in MP)
        # print(f'Found {len(ce_files)} CE files for prefix {file_part}')
        
        return (f, ce_files)
    else:
        # print(f'No CE part found in filename {f}')
        return (f, [f])


files = df['file_name'].unique().tolist()

with mp.Pool(processes=20) as pool:
    results = pool.map(find_ce_group, files)

# 3. Convert list of tuples [(key, val), ...] back to dictionary
ce_available = dict(results)



### Explode and Index

We now map the found CE files to the dataframe and `explode` it. This duplicates the original row for every new CE file found.
* We also extract the specific CE setting (e.g., `-CE20`) into a new column `ms_settings`.
* A unique `annotation_index` is created to track these specific instances.

In [ ]:
# 1. Map the dictionary to a new column
# This creates a column where each cell contains a list, e.g., ['file_CE10.h5', 'file_CE20.h5']
df['ce_expansion'] = df['file_name'].map(ce_available)

# 2. Explode the list column
# This duplicates the original row for every item in the list
df = df.explode('ce_expansion').reset_index(drop=True)

# Optional: If you want the new expanded file to replace the original 'file_name' column
df['file_name'] = df['ce_expansion']
df = df.drop(columns=['ce_expansion'])
def simple_get_ce(f):
    file_parts = os.path.basename(f).split('_')
    ce_part = [part for part in file_parts if '-CE' in part]
    if ce_part:
        return ce_part[0]
    else:
        return None
df['ms_settings'] = df['file_name'].map(simple_get_ce)

df.reset_index(inplace=True,drop=True)
df.index.name = 'annotation_index'
df.reset_index(inplace=True)

print(df.shape)
df.head()

## 5. Extract MS2 Spectra (Parallel Processing)

This is the core data extraction step. We extract the actual mass spectra (MS2) corresponding to our annotated peaks.

**Logic:**
1.  **Read HDF5:** We access the `.h5` version of the file (which allows fast random access compared to raw mzML).
2.  **Filter by m/z:** We look for precursor ions matching the target `mz` within a tolerance (`ppm=10`).
3.  **Filter by RT:** If multiple matches are found, we select the scan closest to the expected retention time (`rt_peak`).
4.  **Extract:** We retrieve the m/z and intensity arrays for that scan.

This is computationally intensive, so it is run in parallel using `multiprocessing`.

In [ ]:

def get_ms2_scan(my_index, filename, polarity, rt_peak, mz, ppm=10):
    new_name = filename.replace('.mzML', '.h5')
    
    # Check if file exists or wrap in try/except if necessary for robustness
    try:
        data = pd.read_hdf(new_name, f'ms2_{polarity[:3]}')
    except (FileNotFoundError, KeyError):
        return None

    data.columns = [c.lower() for c in data.columns]
    
    ppm_error = abs((data['precursor_mz'] - mz) / mz) * 1e6
    data = data[ppm_error < ppm]

    if data.empty:
        return None

    unique_rts = data['rt'].unique()
    closest_rt = min(unique_rts, key=lambda x: abs(x - rt_peak))
    
    data = data[data['rt'] == closest_rt].copy() # .copy() prevents SettingWithCopy warnings later
    data = data[['mz', 'i']]
    data.columns = ['mz', 'intensity']
    data['annotation_index'] = my_index

    return data


tasks = list(zip(
    df['annotation_index'], 
    df['file_name'], 
    df['polarity'], 
    df['rt_peak'], 
    df['mz']
))

with mp.Pool(processes=10) as pool:
    data = pool.starmap(get_ms2_scan, tasks)

data = [res for res in data if res is not None]
data = pd.concat(data, ignore_index=True)


## 6. Aggregate and Save

The extraction results are currently in a "long" format (one row per spectral peak). We aggregate these back into lists of m/z and intensity values per annotation.

* **Group:** Group by `annotation_index`.
* **Merge:** Inner join with the original metadata dataframe (effectively filtering out annotations where no MS2 data was found).
* **Save:** Export the final result to Parquet (efficient for nested column data like lists).

In [ ]:
g = data.groupby('annotation_index').agg({'mz': list, 'intensity': list})
g.rename(columns={'mz':'ms2_mz','intensity':'ms2_intensity'}, inplace=True)
ms2_df = df.merge(g, left_on='annotation_index', right_index=True, how='inner')
print(f'Final number of annotations with MS2 data: {len(ms2_df)}')
parquet_filename = os.path.join(destination_folder, f'ema_standards_{curation_level}_with_ms2.parquet')
ms2_df.to_parquet(parquet_filename)
ms2_df.head()

## 7. Copy Raw Data Files

To create a self-contained dataset, we copy the actual `.mzML` files referenced in our dataset to the `destination_folder`. This ensures the raw data accompanies the metadata.

In [ ]:
# gather the final list of raw files in the dataset
raw_files = df['file_name'].unique().tolist()
# make sure they are all .mzML files
# make it
os.makedirs(destination_folder, exist_ok=True)
for f in raw_files:
    file_to_copy = f.replace('.h5','.mzML')
    dest_file = os.path.join(destination_folder, 'rawdata', os.path.basename(file_to_copy))
    if not os.path.exists(dest_file):
        os.system(f'cp {file_to_copy} {dest_file}')
        
    
        

## 8. Dataset Summary and Statistics

We conclude by generating a comprehensive summary of the curated dataset.

**Steps:**
1.  **Generate InChIKeys:** Convert InChI strings to InChIKeys for robust counting of unique chemical entities.
2.  **Calculate Statistics:** Aggregate counts for files, chromatographies, polarities, collision energies, and adducts.
3.  **Report & Export:** Print a formatted report to the screen and save it as a text file (matching the parquet filename) for documentation.

In [ ]:
from rdkit.Chem import InchiToInchiKey
import textwrap
import os

# 1. Generate InChIKeys (Essential for unique molecule counting)
print("Generating InChIKeys for unique molecule counting...")
ms2_df['inchikey'] = ms2_df['inchi'].apply(lambda x: InchiToInchiKey(x) if pd.notna(x) else None)

# 2. Calculate Statistics
total_spectra = len(ms2_df)
unique_molecules = ms2_df['inchikey'].nunique()
unique_files = ms2_df['file_name'].nunique()
unique_adducts = ms2_df['adduct'].unique()
unique_ces = ms2_df['ms_settings'].dropna().unique()
chromatography_counts = ms2_df['chromatography'].value_counts()
polarity_counts = ms2_df['polarity'].value_counts()

# 3. Construct the Grandiose Summary String
lines = []
lines.append("="*80)
lines.append(f"{'DATASET SUMMARY':^80}")
lines.append("="*80)

lines.append(f"\n{'  DATASET SCALE  ':=^30}")
lines.append(f"Total Annotated MS2 Spectra:    {total_spectra:,}")
lines.append(f"Unique Chemical Entities:       {unique_molecules:,} (by InChIKey)")
lines.append(f"Total Raw Files Referenced:     {unique_files:,}")
if unique_molecules > 0:
    lines.append(f"Avg Spectra per Molecule:       {total_spectra / unique_molecules:.1f}")

lines.append(f"\n{'  EXPERIMENTAL VARIABILITY  ':=^30}")
lines.append(f"Chromatography:")
for method, count in chromatography_counts.items():
    lines.append(f"  • {method.ljust(20)}: {count:,} spectra")

lines.append(f"\nPolarity:")
for pol, count in polarity_counts.items():
    lines.append(f"  • {pol.ljust(20)}: {count:,} spectra")

lines.append(f"\nCollision Energies ({len(unique_ces)} distinct settings):")
ce_str = ", ".join(sorted([str(x) for x in unique_ces]))
lines.append(textwrap.fill(ce_str, width=80, initial_indent="  ", subsequent_indent="  "))

lines.append(f"\nAdducts ({len(unique_adducts)} types):")
adduct_str = ", ".join(sorted([str(x) for x in unique_adducts if pd.notna(x)]))
lines.append(textwrap.fill(adduct_str, width=80, initial_indent="  ", subsequent_indent="  "))

lines.append("\n" + "="*80)

# Join list into a single string
summary_report = "\n".join(lines)

# 4. Print to Screen
print(summary_report)

# 5. Save to Text File
# Matches convention: ema_standards_{curation_level}_with_ms2.parquet
txt_filename = os.path.join(destination_folder, f'ema_standards_{curation_level}_dataset_summary.txt')

with open(txt_filename, 'w') as f:
    f.write(summary_report)

print(f"\nSummary report saved to: {txt_filename}")